In [1]:
from MotorInversion import MotorInversion
import pandas as pd
import importlib
import itertools
from datetime import datetime, date
from io import StringIO
import matplotlib.pyplot as plt

from UniversoActivos import UniversoActivosEstatico, UniversoActivosDinamico
from ProveedorDatos import YFinanceProvider
from VariablesTransformation import FeatureEngineer
import Modelos
Modelos = importlib.reload(Modelos)
import Estrategia
importlib.reload(Estrategia)
from Backtest import BacktestEngine
import requests

RandomForestModel = Modelos.RandomForestModel
XGBoostModel = Modelos.XGBoostModel

EstrategiaMLEquiponderada = Estrategia.EstrategiaMLEquiponderada
EstrategiaMLMinVarAlphaTilt = Estrategia.EstrategiaMLMinVarAlphaTilt
EstrategiaMLMonteCarlo = Estrategia.EstrategiaMLMonteCarlo
EstrategiaMLEquiponderadaMacro = Estrategia.EstrategiaMLEquiponderadaMacro

In [2]:
def get_eurostoxx50_tickers():
    url = 'https://en.wikipedia.org/wiki/EURO_STOXX_50'
    headers = {"User-Agent": "Mozilla/5.0 ..."}
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    tables = pd.read_html(StringIO(response.text), flavor='bs4')
    df = next(t for t in tables if 'Ticker' in t.columns)
    return df['Ticker'].tolist()

tickers = get_eurostoxx50_tickers()
print(len(tickers))
print(tickers)

50
['ADS.DE', 'ADYEN.AS', 'AD.AS', 'AI.PA', 'AIR.PA', 'ALV.DE', 'ABI.BR', 'ARGX.BR', 'ASML.AS', 'CS.PA', 'BAS.DE', 'BAYN.DE', 'BBVA.MC', 'SAN.MC', 'BMW.DE', 'BNP.PA', 'BN.PA', 'DBK.DE', 'DB1.DE', 'DHL.DE', 'DTE.DE', 'ENEL.MI', 'ENI.MI', 'EL.PA', 'RACE.MI', 'RMS.PA', 'IBE.MC', 'ITX.MC', 'IFX.DE', 'INGA.AS', 'ISP.MI', 'OR.PA', 'MC.PA', 'MBG.DE', 'MUV2.DE', 'NDA-FI.HE', 'PRX.AS', 'RHM.DE', 'SAF.PA', 'SGO.PA', 'SAN.PA', 'SAP.DE', 'SU.PA', 'SIE.DE', 'ENR.DE', 'TTE.PA', 'DG.PA', 'UCG.MI', 'VOW.DE', 'WKL.AS']


In [ ]:
universo   = UniversoActivosEstatico(tickers)
fe = FeatureEngineer(criterio=10, ticker_indice="^STOXX50E")
modelo = XGBoostModel(random_state=42, n_splits=3)
estrategia = EstrategiaMLMonteCarlo(modelo=modelo, n_activos_obj=15, umbral_salida=22,
                                    n_simulaciones=500000, peso_min=0.02, peso_max=0.15)

motor = MotorInversion(
    universo       = universo,
    feature_engineer = fe,
    estrategia     = estrategia,
    proveedor_cls  = YFinanceProvider,
    capital_total  = 10000000,
    estado_path    = "./mi_cartera",   # carpeta donde guarda los CSV
    len_ventana = 6
)

In [4]:
# Código para reentrenar el modelo por si cambiamos cosas en el codigo
motor._ultimo_train = None
motor._reentrenar_si_toca(date(2026, 3, 4))
motor._guardar_modelo()

[Train] 2021-03-05 → 2026-02-20 | AUC=0.5484 | {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 150, 'scale_pos_weight': 3, 'subsample': 0.7}


In [ ]:
# Cada viernes ejecutas esto:
señales = motor.ejecutar()

[*********************100%***********************]  51 of 51 completed
[*********************100%***********************]  51 of 51 completed


Compras hechas a fecha 2026-03-13 00:00:00
  Ticker Accion  Cantidad      Precio     CT  Precio_Ejecutado
 ASML.AS COMPRA       564 1180.400024 1.1804         1181.5804
  ENR.DE COMPRA      4362  152.649994 0.1526          152.8026
 BBVA.MC COMPRA     36623   18.184999 0.0182           18.2032
  ADS.DE COMPRA      4716  141.199997 0.1412          141.3412
  PRX.AS COMPRA     14659   45.430000 0.0454           45.4754
  SAP.DE COMPRA      3987  167.020004 0.1670          167.1870
   EL.PA COMPRA      3159  210.800003 0.2108          211.0108
 RACE.MI COMPRA      2276  292.600006 0.2926          292.8926
  WKL.AS COMPRA      9919   67.139999 0.0671           67.2071
 BAYN.DE COMPRA     17020   39.130001 0.0391           39.1691
ADYEN.AS COMPRA       726  917.299988 0.9173          918.2173
  DBK.DE COMPRA     25909   25.705000 0.0257           25.7307
  RHM.DE COMPRA       429 1550.500000 1.5505         1552.0505
 ARGX.BR COMPRA      1081  616.000000 0.6160          616.6160
  AIR.PA COM

Viernes 20 de marzo de 2026

In [5]:
fecha = date(2026, 3, 20)
señales = motor.ejecutar(fecha)

c:\Users\jpuerta\OneDrive - Analistas Financieros Internacionales (Afi)\Documentos\Master Quant\Gestión de activos\Trabajo gestión cuantitativa\Trabajo_gestion_cuantitativa\ProveedorDatos.py:72: FutureWarning: DataFrameGroupBy.resample operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly = weekly.agg(


  Ticker Accion  Cantidad      Precio       CT  Precio_Ejecutado
   EL.PA COMPRA        74  194.750000 0.097375        194.847375
ADYEN.AS COMPRA         3  863.500000 0.431750        863.931750
  PRX.AS COMPRA      1077   40.020000 0.020010         40.040010
  ENR.DE COMPRA       112  140.750000 0.070375        140.820375
  ADS.DE COMPRA         1  133.500000 0.066750        133.566750
 RACE.MI COMPRA        24  273.799988 0.136900        273.936888
  SAP.DE COMPRA       107  153.820007 0.076910        153.896917
  RMS.PA COMPRA       380 1656.000000 0.828000       1656.828000
  ENI.MI COMPRA     26650   23.620001 0.011810         23.631811
  SGO.PA COMPRA      9246   68.080002 0.034040         68.114042
  WKL.AS  VENTA       294   65.440002 0.032720         65.407282
 ARGX.BR  VENTA         4  584.799988 0.292400        584.507588
 BAYN.DE  VENTA       612   38.384998 0.019192         38.365806
  DBK.DE  VENTA       472   24.760000 0.012380         24.747620
 BBVA.MC  VENTA     36623

Viernes 27 de marzo de 2026

In [ ]:
fecha = date(2026, 3, 27)
señales = motor.ejecutar(fecha)

  Ticker Accion  Cantidad      Precio       CT  Precio_Ejecutado
  IFX.DE COMPRA      5846   37.430000 0.018715         37.448715
  ENI.MI COMPRA     34714   23.924999 0.011962         23.936962
 ASML.AS COMPRA       279 1146.400024 0.573200       1146.973224
  RHM.DE COMPRA       221 1379.500000 0.689750       1380.189750
  TTE.PA COMPRA     18700   78.489998 0.039245         78.529243
  UCG.MI COMPRA     12173   60.220001 0.030110         60.250111
  SAF.PA COMPRA      1724  278.399994 0.139200        278.539194
 INGA.AS COMPRA     34709   21.719999 0.010860         21.730859
  SAN.MC COMPRA     56041    9.402000 0.004701          9.406701
 BAYN.DE  VENTA      1609   38.259998 0.019130         38.240868
  SGO.PA  VENTA      1553   69.120003 0.034560         69.085443
  DBK.DE  VENTA     13938   24.915001 0.012458         24.902543
  ADS.DE  VENTA      2845  132.050003 0.066025        131.983978
  PRX.AS  VENTA     15736   38.720001 0.019360         38.700641
   EL.PA  VENTA      3233

Viernes 3 de abril de 2026

In [8]:
fecha = date(2026, 4, 3)
señales = motor.ejecutar(fecha)


1 Failed download:
['ABI.BR']: OperationalError('database is locked')


[Señales] 2026-04-03 | Pesos objetivo: {'ADS.DE': 0.039212258605359214, 'ADYEN.AS': 0.025893622067794997, 'ASML.AS': 0.13792986724016687, 'BAYN.DE': 0.09055288399939496, 'BBVA.MC': 0.07347758338918811, 'DBK.DE': 0.03365522007560314, 'ENI.MI': 0.12345330697614293, 'ENR.DE': 0.02612472421507548, 'IFX.DE': 0.04816259175438095, 'INGA.AS': 0.06813066522371464, 'RACE.MI': 0.02383311062131374, 'SAF.PA': 0.05836475231588556, 'SAN.MC': 0.05241753137401491, 'SGO.PA': 0.02786433914828268, 'TTE.PA': 0.17092754299368176}
Valor de la cartera antes de ejecutar señales: 9701263.39€
  Ticker Accion  Cantidad      Precio       CT  Precio_Ejecutado
 BAYN.DE COMPRA      7328   39.695000 0.019847         39.714847
 BBVA.MC COMPRA     37957   18.770000 0.009385         18.779385
 RACE.MI COMPRA       782  295.500000 0.147750        295.647750
  SAF.PA COMPRA       246  287.299988 0.143650        287.443638
 ASML.AS COMPRA       314 1161.000000 0.580500       1161.580500
  ADS.DE COMPRA       947  134.899994

Viernes 10 de abril de 2026

In [5]:
fecha = date(2026, 4, 10)
señales = motor.ejecutar(fecha)

[Señales] 2026-04-10 | Pesos objetivo: {'ADS.DE': 0.05005870125734341, 'ADYEN.AS': 0.031765321164300295, 'ASML.AS': 0.061524215320116, 'BAYN.DE': 0.07995083906160282, 'BBVA.MC': 0.03320283022735842, 'DBK.DE': 0.02737977846630994, 'ENI.MI': 0.14630698079463786, 'ENR.DE': 0.1538532063280711, 'IFX.DE': 0.025589251754711082, 'PRX.AS': 0.03589832533053728, 'RACE.MI': 0.03018889388409957, 'SAF.PA': 0.07433886640498791, 'SAN.MC': 0.052628888678880124, 'SGO.PA': 0.04346069499897298, 'TTE.PA': 0.1538532063280711}
Valor de la cartera antes de ejecutar señales: 10089519.21€
  Ticker Accion  Cantidad      Precio       CT  Precio_Ejecutado
  PRX.AS COMPRA      2584   41.555000 0.020778         41.575778
  SAN.MC COMPRA     19597   10.516000 0.005258         10.521258
  ENR.DE COMPRA      6951  167.220001 0.083610        167.303611
  DBK.DE COMPRA       663   27.709999 0.013855         27.723854
  ADS.DE COMPRA      1900  137.800003 0.068900        137.868903
  TTE.PA COMPRA      3599   78.610001 0.

### Viernes 17 de abril de 2026

In [4]:
fecha = date(2026, 4, 17)
señales = motor.ejecutar(fecha)

[Señales] 2026-04-17 | Pesos objetivo: {'ADS.DE': 0.03267702723433555, 'ADYEN.AS': 0.0283878306535348, 'ASML.AS': 0.14274182325003448, 'BAYN.DE': 0.1590946875195419, 'BBVA.MC': 0.03568916574814934, 'DBK.DE': 0.048083433207925415, 'ENI.MI': 0.13026643837916066, 'ENR.DE': 0.08575024006483886, 'ITX.MC': 0.02349302812410177, 'PRX.AS': 0.03458663342385433, 'SAF.PA': 0.03878563119694552, 'SAP.DE': 0.021403213808380783, 'SGO.PA': 0.032162574302530944, 'TTE.PA': 0.1590946875195419, 'WKL.AS': 0.02778358556712374}
Valor de la cartera antes de ejecutar señales: 10138858.22€
  Ticker Accion  Cantidad      Precio       CT  Precio_Ejecutado
  TTE.PA COMPRA      2329   73.070000 0.036535         73.106535
  DBK.DE COMPRA      6891   28.910000 0.014455         28.924455
  WKL.AS COMPRA      3946   71.339996 0.035670         71.375666
  ITX.MC COMPRA      4327   55.020000 0.027510         55.047510
  SAP.DE COMPRA      1388  156.240005 0.078120        156.318125
 BAYN.DE COMPRA     19134   41.099998 0.

### Viernes 24 de abril de 2026

In [4]:
fecha = date(2026, 4, 24)
señales = motor.ejecutar(fecha)

[Señales] 2026-04-24 | Pesos objetivo: {'ADS.DE': 0.022843281131737513, 'ADYEN.AS': 0.04542920719879166, 'ASML.AS': 0.1307452824136683, 'BAYN.DE': 0.045035829406905585, 'BBVA.MC': 0.05075746108479951, 'DBK.DE': 0.02528443103205414, 'ENI.MI': 0.16190914806933293, 'ENR.DE': 0.16190914806933293, 'IFX.DE': 0.0599503055677034, 'ITX.MC': 0.1454021044187801, 'PRX.AS': 0.04198775142263562, 'RHM.DE': 0.029545597944379016, 'SAP.DE': 0.023389152183664056, 'SGO.PA': 0.022741378651775578, 'WKL.AS': 0.03306992140443948}
Valor de la cartera antes de ejecutar señales: 10053617.44€
  Ticker Accion  Cantidad      Precio       CT  Precio_Ejecutado
  PRX.AS COMPRA      2251   41.535000 0.020767         41.555767
  ENI.MI COMPRA     10225   22.950001 0.011475         22.961476
 BBVA.MC COMPRA      9644   18.580000 0.009290         18.589290
  SAP.DE COMPRA       208  147.279999 0.073640        147.353639
  ITX.MC COMPRA     23473   52.560001 0.026280         52.586281
  WKL.AS COMPRA      1059   66.419998 

### Viernes 1 de mayo de 2026

In [4]:
fecha = date(2026, 5, 1)
señales = motor.ejecutar(fecha)

[Señales] 2026-05-01 | Pesos objetivo: {'ADYEN.AS': 0.02334821576280248, 'ASML.AS': 0.151569641670233, 'BAYN.DE': 0.133838768462271, 'BBVA.MC': 0.07185293292155927, 'BNP.PA': 0.036415211157781985, 'DBK.DE': 0.06377503574764606, 'ENI.MI': 0.151569641670233, 'ENR.DE': 0.15106697525069432, 'IFX.DE': 0.06146470749323522, 'PRX.AS': 0.02137373202272394, 'RHM.DE': 0.02935884459899176, 'RMS.PA': 0.021079772512255297, 'SAN.MC': 0.036584768256791654, 'SAP.DE': 0.021685005447796675, 'WKL.AS': 0.02501674702498431}
Valor de la cartera antes de ejecutar señales: 10063211.83€
  Ticker Accion  Cantidad      Precio       CT  Precio_Ejecutado
  DBK.DE COMPRA     11411   26.500000 0.013250         26.513250
  SAN.MC COMPRA     14533   10.380000 0.005190         10.385190
  RMS.PA COMPRA       130 1623.500000 0.811750       1624.311750
  BNP.PA COMPRA      4104   89.230003 0.044615         89.274618
  ENR.DE COMPRA       380  180.580002 0.090290        180.670292
  PRX.AS COMPRA      1204   41.070000 0.02

### Viernes 8 de mayo de 2026

In [4]:
fecha = date(2026, 5, 8)
señales = motor.ejecutar(fecha)

[Señales] 2026-05-08 | Pesos objetivo: {'ADS.DE': 0.029186169139278465, 'ADYEN.AS': 0.036699799788299604, 'ASML.AS': 0.1424441223230154, 'BAYN.DE': 0.09019757646929229, 'BBVA.MC': 0.022621024966007503, 'DBK.DE': 0.030182566158588598, 'ENI.MI': 0.1618748059908376, 'ENR.DE': 0.135647521358989, 'IFX.DE': 0.11876528891679891, 'PRX.AS': 0.031057148578679958, 'RHM.DE': 0.03125360921477971, 'RMS.PA': 0.026476260965747987, 'SAN.MC': 0.05993879076677352, 'SAP.DE': 0.038043539727949174, 'WKL.AS': 0.04561177563496227}
Valor de la cartera antes de ejecutar señales: 10179695.42€
  Ticker Accion  Cantidad      Precio       CT  Precio_Ejecutado
  ADS.DE COMPRA      2022  146.850006 0.073425        146.923431
  RMS.PA COMPRA        32 1661.000000 0.830500       1661.830500
  WKL.AS COMPRA      3709   61.900002 0.030950         61.930952
  PRX.AS COMPRA      2471   41.009998 0.020505         41.030503
  IFX.DE COMPRA      8778   61.660000 0.030830         61.690830
  SAN.MC COMPRA     22827   10.466000

### Viernes 15 de mayo de 2026

In [ ]:
fecha = date(2026, 5, 15)
señales = motor.ejecutar(fecha)

In [5]:
import yfinance as yf

# Definir el ticker (ejemplo: Apple)
ticker = "IFX.DE"

# Descargar datos con periodo de 1 día e intervalo de 1 minuto
data = yf.download(ticker, period="5d", interval="1d")

# Mostrar las últimas filas
print(data.tail())

[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open   Volume
Ticker         IFX.DE     IFX.DE     IFX.DE     IFX.DE   IFX.DE
Date                                                           
2026-04-24  54.150002  54.869999  52.700001  52.700001  7879354
2026-04-27  53.599998  55.419998  53.279999  55.000000  5228822
2026-04-28  52.799999  54.200001  51.540001  52.919998  5526296
2026-04-29  55.700001  56.340000  53.400002  53.480000  5751825
2026-04-30  57.130001  57.299999  55.070000  55.299999  5641309
